# Part C — aggregations + plots

Load `events.parquet` (written by Part B), parse `created_at` to a proper datetime, then:
- **C1** — top-10 users by push-event count and total commit count → `top_users.png`
- **C2** — top-10 repos by all-event count and by commit count (PushEvent only) → `top_repos.png`
- **C3** — hourly + day-of-week activity patterns for the busiest repo → `hourly_pattern.png`, `dow_pattern.png`

In [1]:
import os
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')   # headless backend — saves PNGs without a display
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

for extension_name in ('pandas.period', 'pandas.interval', 'pandas.arrow_dtype'):
    try:
        pa.unregister_extension_type(extension_name)
    except (KeyError, pa.ArrowKeyError):
        pass

PARQUET = 'events.parquet'

if not os.path.exists(PARQUET):
    raise FileNotFoundError(
        f"Missing {PARQUET!r} in {os.getcwd()}. Run 02_to_parquet.ipynb first.")

## Load Parquet + parse timestamps

In [2]:
start = time.perf_counter()
table = pq.read_table(PARQUET)
df = table.to_pandas()
df['created_at'] = pd.to_datetime(df['created_at'], utc=True)
elapsed_load = time.perf_counter() - start

print(f'Loaded {len(df):,} rows in {elapsed_load:.2f} s')
print(df.dtypes)
df.head(3)

Loaded 28,506,909 rows in 58.92 s
event_id                     object
event_type                 category
actor_login                  object
repo_name                    object
created_at      datetime64[us, UTC]
commit_count                  int32
dtype: object


,event_id,event_type,actor_login,repo_name,created_at,commit_count
0,2594237218,IssueCommentEvent,yongli-d,yongli-d/StravaBuddy,2015-02-20 01:00:00+00:00,0
1,2594237220,CreateEvent,vgeshel,yummly/s3-to-redshift,2015-02-20 01:00:01+00:00,0
2,2594237222,PushEvent,davidcarlsonberg,PubWlkr/PubWlkr,2015-02-20 01:00:01+00:00,1


## C1 — top-10 users by push-event count and commit count

Filter to `PushEvent` rows only, then rank the top 10 `actor_login` values
(a) by number of push events and (b) by total `commit_count`.  
Both rankings are shown as horizontal bar charts side-by-side in one 14×5 figure.

In [3]:
pushes = df[df['event_type'] == 'PushEvent'].copy()

# (a) top 10 by push-event count
start_c1a = time.perf_counter()
top_users_events = (
    pushes.groupby('actor_login', observed=True)
          .size()
          .nlargest(10)
          .sort_values()          # ascending so the longest bar is at the top
)
elapsed_c1a = time.perf_counter() - start_c1a
print(f'C1a (top users by event count) — {elapsed_c1a:.3f} s')
print(top_users_events)

C1a (top users by event count) — 7.480 s
actor_login
diversify-exp-user     18872
pbaffiliate1           19814
asfgit                 22022
qdm                    22113
chapuni                25662
meatballhat            32325
greatfire              47191
greatfirebot           52095
mirror-updates         57241
KenanSulayman         105427
dtype: int64


In [4]:
# (b) top 10 by commit count
start_c1b = time.perf_counter()
top_users_commits = (
    pushes.groupby('actor_login', observed=True)['commit_count']
          .sum()
          .nlargest(10)
          .sort_values()          # ascending so the longest bar is at the top
)
elapsed_c1b = time.perf_counter() - start_c1b
print(f'C1b (top users by commit count) — {elapsed_c1b:.3f} s')
print(top_users_commits)

C1b (top users by commit count) — 7.906 s
actor_login
sharoonthomas        111511
gbtami               118755
gugod                120998
thierryreding        144692
pzia                 148425
birkenfeld           247933
greatfire-martin     313820
i5o                  603229
greatfire           1459064
mirror-updates      9417212
Name: commit_count, dtype: int32


In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top_users_events.index, top_users_events.values, color='steelblue')
ax1.set_xlabel('Number of PushEvents')
ax1.set_ylabel('Actor login')
ax1.set_title('Top 10 Users — Push-Event Count')
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2.barh(top_users_commits.index, top_users_commits.values, color='darkorange')
ax2.set_xlabel('Total Commits')
ax2.set_ylabel('Actor login')
ax2.set_title('Top 10 Users — Total Commit Count')
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('top_users.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved top_users.png')

Saved top_users.png


C:\Users\elyas\AppData\Local\Temp\ipykernel_22604\1671524412.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## C2 — top-10 repos by all-event count and by commit count

(a) All event types, grouped by `repo_name`.  
(b) PushEvents only, summed `commit_count` per `repo_name`.  
The repo ranked #1 in (a) is stored as `REPO_BUSIEST` for use in C3.

In [6]:
# (a) all events — any type
start_c2a = time.perf_counter()
top_repos_events = (
    df.groupby('repo_name', observed=True)
      .size()
      .nlargest(10)
      .sort_values()
)
elapsed_c2a = time.perf_counter() - start_c2a

REPO_BUSIEST = top_repos_events.idxmax()   # repo ranked #1 by total event count
print(f'C2a (top repos by event count) — {elapsed_c2a:.3f} s')
print(f'REPO_BUSIEST = {REPO_BUSIEST!r}')
print(top_repos_events)

C2a (top repos by event count) — 21.080 s
REPO_BUSIEST = 'KenanSulayman/heartbeat'
repo_name
greatfire/feeds             19920
qdm/qdm.github.io           22113
owncloud/core               22899
apache/spark                25169
rust-lang/rust              28927
sakai-mirror/melete         40827
direwolf-github/my-app      46511
chrsmith/bwapi              50534
greatfire/wiki              61781
KenanSulayman/heartbeat    105335
dtype: int64


In [7]:
# (b) PushEvents only — total commit_count per repo
start_c2b = time.perf_counter()
top_repos_commits = (
    pushes.groupby('repo_name', observed=True)['commit_count']
          .sum()
          .nlargest(10)
          .sort_values()
)
elapsed_c2b = time.perf_counter() - start_c2b
print(f'C2b (top repos by commit count) — {elapsed_c2b:.3f} s')
print(top_repos_commits)

C2b (top repos by commit count) — 10.679 s
repo_name
g0v-data/mirror           120664
thierryreding/linux       139678
v-l-m/vlm                 147883
odoo-dev/odoo             236192
sphinx-doc/sphinx         247537
i5o/committer             602015
sakai-mirror/ambrosia    1047970
greatfire/z              1739965
sakai-mirror/mneme       3977237
sakai-mirror/melete      4367422
Name: commit_count, dtype: int32


In [8]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top_repos_events.index, top_repos_events.values, color='teal')
ax1.set_xlabel('Total Events (all types)')
ax1.set_ylabel('Repository')
ax1.set_title('Top 10 Repos — All-Event Count')
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2.barh(top_repos_commits.index, top_repos_commits.values, color='purple')
ax2.set_xlabel('Total Commits (PushEvent only)')
ax2.set_ylabel('Repository')
ax2.set_title('Top 10 Repos — Total Commit Count')
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('top_repos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved top_repos.png')

Saved top_repos.png


C:\Users\elyas\AppData\Local\Temp\ipykernel_22604\3898156687.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## C3 — time-series patterns for the busiest repo

Filter to `repo_name == REPO_BUSIEST`, then produce two charts:
- **Hourly pattern** (0–23): event count per hour of day, peak hour marked with a dashed vertical line → `hourly_pattern.png`
- **Day-of-week pattern** (Mon–Sun, pandas convention 0–6): event count per weekday, weekend bars (Sat=5, Sun=6) highlighted → `dow_pattern.png`

In [9]:
df_repo = df[df['repo_name'] == REPO_BUSIEST].copy()
print(f'Rows for {REPO_BUSIEST!r}: {len(df_repo):,}')

Rows for 'KenanSulayman/heartbeat': 105,335


In [10]:
# Hourly pattern (0–23)
start_c3h = time.perf_counter()
hourly = df_repo.groupby(df_repo['created_at'].dt.hour).size().reindex(range(24), fill_value=0)
elapsed_c3h = time.perf_counter() - start_c3h

peak_hour = int(hourly.idxmax())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
ax.axvline(peak_hour, color='red', linestyle='--', linewidth=1.5, label=f'Peak: {peak_hour:02d}:00')
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Number of Events')
ax.set_title(f'Hourly Activity Pattern — {REPO_BUSIEST}')
ax.set_xticks(range(24))
ax.legend()
plt.tight_layout()
plt.savefig('hourly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'C3-hourly — {elapsed_c3h:.3f} s   peak hour: {peak_hour:02d}:00 UTC')
print('Saved hourly_pattern.png')

C3-hourly — 0.012 s   peak hour: 01:00 UTC
Saved hourly_pattern.png


C:\Users\elyas\AppData\Local\Temp\ipykernel_22604\1326756266.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Day-of-week pattern — pandas dayofweek: 0=Monday … 6=Sunday
# Weekend: Saturday (5) and Sunday (6) highlighted in coral.
start_c3d = time.perf_counter()
dow = df_repo.groupby(df_repo['created_at'].dt.dayofweek).size().reindex(range(7), fill_value=0)
elapsed_c3d = time.perf_counter() - start_c3d

DAY_LABELS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
WEEKEND    = {5, 6}   # Sat, Sun in pandas convention
colors     = ['coral' if d in WEEKEND else 'steelblue' for d in range(7)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(7), dow.values, color=colors, edgecolor='white')
ax.set_xticks(range(7))
ax.set_xticklabels(DAY_LABELS)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Number of Events')
ax.set_title(f'Day-of-Week Activity Pattern — {REPO_BUSIEST}')

# Legend patches
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='steelblue', label='Weekday'),
    Patch(color='coral',     label='Weekend'),
])
plt.tight_layout()
plt.savefig('dow_pattern.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'C3-dow — {elapsed_c3d:.3f} s')
print('Saved dow_pattern.png')

C3-dow — 0.019 s
Saved dow_pattern.png


C:\Users\elyas\AppData\Local\Temp\ipykernel_22604\1324282635.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part C timing summary

In [12]:
print('=== Part C — wall-clock timing summary ===')
print(f'Parquet load + timestamp parse : {elapsed_load:.2f} s')
print(f'C1a — top users by event count : {elapsed_c1a:.3f} s')
print(f'C1b — top users by commit count: {elapsed_c1b:.3f} s')
print(f'C2a — top repos by event count : {elapsed_c2a:.3f} s')
print(f'C2b — top repos by commit count: {elapsed_c2b:.3f} s')
print(f'C3  — hourly aggregation       : {elapsed_c3h:.3f} s')
print(f'C3  — day-of-week aggregation  : {elapsed_c3d:.3f} s')

=== Part C — wall-clock timing summary ===
Parquet load + timestamp parse : 58.92 s
C1a — top users by event count : 7.480 s
C1b — top users by commit count: 7.906 s
C2a — top repos by event count : 21.080 s
C2b — top repos by commit count: 10.679 s
C3  — hourly aggregation       : 0.012 s
C3  — day-of-week aggregation  : 0.019 s
